<a href="https://colab.research.google.com/github/RubaEgbaria/RAG/blob/master/chatgpt_api_test_heldout_qa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ChatGPT API Evaluation on Held-Out QA Test Split

This notebook evaluates a ChatGPT/OpenAI model on the same held-out `0.09` split used in encoder-decoder QA project.

It loads `data_qa.csv`, recreates the same split with `seed=3407`, sends the test questions to the OpenAI API, saves predictions incrementally, and computes BLEU, BERTScore, Semantic Textual Similarity, ROUGE-L, and ROUGE-S.

In [14]:
# Install if needed
# !pip install -q openai datasets sacrebleu bert-score sentence-transformers rouge-score pandas tqdm torch

## 1. Imports and configuration

In [2]:
import os
import re
import time
import random
import getpass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import sacrebleu

from datasets import Dataset
from tqdm import tqdm
from bert_score import score as bertScore
from sentence_transformers import SentenceTransformer, util
from rouge_score import rouge_scorer

from openai import OpenAI
from openai import RateLimitError, APITimeoutError, APIConnectionError, APIError

seed = 3407
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

dataPath = "data_qa.csv"
testSize = 0.09

# Use None for the full held-out test set. Use 20/100 first to control cost.
maxExamples = None

# Change this if your API account uses another available model.
openaiModelName = "gpt-5.4"

maxOutputTokens = 256
temperature = 0

outputDir = Path("./openai_chatgpt_evaluation")
outputDir.mkdir(parents=True, exist_ok=True)

safeModelName = openaiModelName.replace("/", "_").replace(":", "_")
predictionsPath = outputDir / f"{safeModelName}_heldout_predictions.csv"
metricsPath = outputDir / f"{safeModelName}_heldout_metrics.csv"

deviceForMetrics = "cuda" if torch.cuda.is_available() else "cpu"

print("Data path:", dataPath)
print("Model:", openaiModelName)
print("Max examples:", maxExamples)
print("Metrics device:", deviceForMetrics)
print("Predictions path:", predictionsPath)
print("Metrics path:", metricsPath)

Data path: data_qa.csv
Model: gpt-5.4
Max examples: None
Metrics device: cpu
Predictions path: openai_chatgpt_evaluation/gpt-5.4_heldout_predictions.csv
Metrics path: openai_chatgpt_evaluation/gpt-5.4_heldout_metrics.csv


## 2.OpenAI API key

In [3]:
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OPENAI_API_KEY: ")

client = OpenAI()
print("OpenAI client created.")

Enter OPENAI_API_KEY: ··········
OpenAI client created.


## 4. Load data and recreate the same 0.09 held-out split

In [4]:
df = pd.read_csv(dataPath).fillna("")

if "ref_link" in df.columns:
    df["ref_link"] = df["ref_link"].replace(
        "https://youtu.be/IQ5RDN_S25w",
        "https://youtu.be/GFjA6efoSHY"
    )
    df["ref_link"] = df["ref_link"].replace(
        "https://youtu.be/Tk_sqMBOpY4",
        ""
    )

requiredColumns = ["instruction", "question", "answer"]
missingColumns = [column for column in requiredColumns if column not in df.columns]
if missingColumns:
    raise ValueError(f"Missing required columns: {missingColumns}")

useContext = False

def buildInput(row):
    basePrompt = f"""أجب عن السؤال التالي باللغة العربية.

التعليمة:
{row["instruction"]}

السؤال:
{row["question"]}"""

    if useContext and "context_text" in row and str(row["context_text"]).strip():
        basePrompt += f"""

السياق:
{row["context_text"]}"""

    return basePrompt

df["inputText"] = df.apply(buildInput, axis=1)
df["targetText"] = df["answer"].astype(str)

dataset = Dataset.from_pandas(df[["inputText", "targetText"]])
splitDataset = dataset.train_test_split(test_size=testSize, seed=seed)

trainRawDataset = splitDataset["train"]
evalRawDataset = splitDataset["test"]

evalDf = pd.DataFrame({
    "rowIndex": list(range(len(evalRawDataset))),
    "inputText": evalRawDataset["inputText"],
    "referenceAnswer": evalRawDataset["targetText"]
})

if maxExamples is not None:
    evalDf = evalDf.head(maxExamples).copy()

inputTexts = evalDf["inputText"].astype(str).tolist()
references = evalDf["referenceAnswer"].astype(str).tolist()

print("Full train size:", len(trainRawDataset))
print("Full test size:", len(splitDataset["test"]))
print("Evaluation size used in this run:", len(evalDf))
print("\\nSample input:\\n", inputTexts[0][:700])
print("\\nSample reference:\\n", references[0][:500])

Full train size: 15875
Full test size: 1571
Evaluation size used in this run: 1571
\nSample input:\n أجب عن السؤال التالي باللغة العربية.

التعليمة:
أنت راوٍ وحارس للذاكرة الجماعية الفلسطينية، متخصص في روايات القرى والمدن الفلسطينيه. أجب بناءً على الشهادات الشفوية والمصادر المحلية الموثّقة، مع الحفاظ على أصالة السياق الإنساني.

السؤال:
كيف عرقلت قوات الاحتلال وصول طلبة اللبن الشرقية إلى مدارسهم؟
\nSample reference:\n منعتهم من السير على خط نابلس رام الله الرئيسي وأجبرتهم على الالتفاف عبر طرق وعرة وطويلة تصل إلى ثلاثة كيلومترات.

هذا الجواب مستند إلى المصدر التالي: https://www.howiyya.com/VwHomenewsView?start=373


## 5. Prompt template

In [5]:
def buildChatGptPrompt(inputText):
    return f"""أجب عن السؤال التالي باللغة العربية فقط.

تعليمات:
- أجب بإجابة مباشرة ومختصرة.
- لا تخترع روابط أو مصادر.
- إذا لم تكن المعلومة متوفرة لديك، اكتب: لا تتوفر معلومات كافية.

النص:
{inputText}

الإجابة:"""

## 6. API call

In [6]:
def extractOpenAiText(response):
    if hasattr(response, "output_text") and response.output_text:
        return response.output_text.strip()

    texts = []
    if hasattr(response, "output") and response.output:
        for item in response.output:
            if hasattr(item, "content"):
                for content in item.content:
                    if hasattr(content, "text"):
                        texts.append(content.text)

    return "\\n".join(texts).strip()

def callOpenAiOnce(prompt):
    response = client.responses.create(
        model=openaiModelName,
        input=[
            {
                "role": "system",
                "content": (
                    "أنت نظام إجابة باللغة العربية. "
                    "أجب فقط عن السؤال المعطى، ولا تضف مصادر غير موجودة."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=temperature,
        max_output_tokens=maxOutputTokens
    )
    return extractOpenAiText(response)

def callOpenAiWithRetry(prompt, maxRetries=5, baseSleepSeconds=2):
    lastError = None

    for attempt in range(maxRetries):
        try:
            prediction = callOpenAiOnce(prompt)
            return prediction, ""

        except (RateLimitError, APITimeoutError, APIConnectionError, APIError) as error:
            lastError = error
            sleepSeconds = baseSleepSeconds * (2 ** attempt)
            print(f"Retryable API error on attempt {attempt + 1}/{maxRetries}: {type(error).__name__}. Sleeping {sleepSeconds}s.")
            time.sleep(sleepSeconds)

        except Exception as error:
            lastError = error
            break

    return "", f"{type(lastError).__name__}: {str(lastError)}"

## 7.test on 5 examples

In [7]:
smokeRows = evalDf.head(5).copy()

for _, row in smokeRows.iterrows():
    prompt = buildChatGptPrompt(row["inputText"])
    prediction, error = callOpenAiWithRetry(prompt, maxRetries=2)

    print("=" * 100)
    print("ROW:", row["rowIndex"])
    print("ERROR:", error)
    print("\\nREFERENCE:")
    print(row["referenceAnswer"][:500])
    print("\\nPREDICTION:")
    print(prediction)

ROW: 0
ERROR: 
\nREFERENCE:
منعتهم من السير على خط نابلس رام الله الرئيسي وأجبرتهم على الالتفاف عبر طرق وعرة وطويلة تصل إلى ثلاثة كيلومترات.

هذا الجواب مستند إلى المصدر التالي: https://www.howiyya.com/VwHomenewsView?start=373
\nPREDICTION:
عرقلت قوات الاحتلال وصول طلبة اللبن الشرقية إلى مدارسهم عبر إغلاق الطرق والحواجز العسكرية، ومضايقتهم أثناء المرور، ما كان يؤخرهم أو يمنعهم من الوصول إلى المدرسة.
ROW: 1
ERROR: 
\nREFERENCE:
غادرت العائلة إثر النكبة، فتوزع الأبناء بين الأردن وسوريا والعراق.

هذا الجواب مستند إلى المصدر التالي: https://www.howiyya.com/VwFamiliesStoriesView?start=1706
\nPREDICTION:
لا تتوفر معلومات كافية.
ROW: 2
ERROR: 
\nREFERENCE:
أعدم مستوطنون اليوم السبت18/10/2014، عشرات أشجار الزيتون في قرية الجبعة جنوب غرب بيت لحم.

هذا الجواب مستند إلى المصدر التالي: https://www.howiyya.com/VwHomenewsView?start=1050
\nPREDICTION:
المكان المرتبط بالحدث هو: جنوب بيت لحم.
ROW: 3
ERROR: 
\nREFERENCE:
ليس هذا فحسب، فكلاهما يقيمان في مركزي "إيواء"، أقامتهما وكالة غوث وتشغيل اللاجئين ا

## 8. Full prediction run with resume support

In [9]:
def loadCompletedPredictions(path):
    if not path.exists():
        return {}

    previousDf = pd.read_csv(path).fillna("")
    if "rowIndex" not in previousDf.columns:
        return {}

    completed = {}
    for _, row in previousDf.iterrows():
        completed[int(row["rowIndex"])] = {
            "prediction": str(row.get("prediction", "")),
            "error": str(row.get("error", ""))
        }

    return completed

def savePredictionRows(rows, path):
    tempDf = pd.DataFrame(rows).sort_values("rowIndex")
    tempDf.to_csv(path, index=False, encoding="utf-8-sig")

completed = loadCompletedPredictions(predictionsPath)
allRows = []

for _, row in tqdm(evalDf.iterrows(), total=len(evalDf)):
    rowIndex = int(row["rowIndex"])

    if rowIndex in completed and completed[rowIndex]["prediction"].strip():
        prediction = completed[rowIndex]["prediction"]
        error = completed[rowIndex]["error"]
    else:
        prompt = buildChatGptPrompt(row["inputText"])
        prediction, error = callOpenAiWithRetry(prompt)

    allRows.append({
        "rowIndex": rowIndex,
        "inputText": row["inputText"],
        "referenceAnswer": row["referenceAnswer"],
        "prediction": prediction,
        "error": error,
        "model": openaiModelName
    })

    savePredictionRows(allRows, predictionsPath)

print("Saved predictions to:", predictionsPath)
resultsDf = pd.read_csv(predictionsPath).fillna("").sort_values("rowIndex")
resultsDf.head()

100%|██████████| 1571/1571 [51:23<00:00,  1.96s/it]

Saved predictions to: openai_chatgpt_evaluation/gpt-5.4_heldout_predictions.csv


,rowIndex,inputText,referenceAnswer,prediction,error,model
0,0,أجب عن السؤال التالي باللغة العربية.\n\nالتعلي...,منعتهم من السير على خط نابلس رام الله الرئيسي ...,عرقلت قوات الاحتلال وصول طلبة اللبن الشرقية إل...,,gpt-5.4
1,1,أجب عن السؤال التالي باللغة العربية.\n\nالتعلي...,غادرت العائلة إثر النكبة، فتوزع الأبناء بين ال...,لا تتوفر معلومات كافية.,,gpt-5.4
2,2,أجب عن السؤال التالي باللغة العربية.\n\nالتعلي...,أعدم مستوطنون اليوم السبت18/10/2014، عشرات أشج...,المكان المرتبط بالحدث هو: جنوب بيت لحم.,,gpt-5.4
3,3,أجب عن السؤال التالي باللغة العربية.\n\nالتعلي...,"ليس هذا فحسب، فكلاهما يقيمان في مركزي ""إيواء""،...",استهدف الاحتلال في مخيم الشاطئ منشآت مدنية، بي...,,gpt-5.4
4,4,أجب عن السؤال التالي باللغة العربية.\n\nالتعلي...,درس في مدرسة المنزول ومدرسة تين عزيز، ثم ذهب إ...,لا تتوفر معلومات كافية.,,gpt-5.4


## 9. Metrics helper functions

In [10]:
def normalizeText(text):
    return str(text).strip()

def cleanGeneratedText(text):
    text = str(text)
    text = re.sub(r"<extra_id_\\d+>", " ", text)
    text = text.replace("<pad>", " ")
    text = text.replace("</s>", " ")
    text = re.sub(r"\\s+([،؛؟.!:])", r"\\1", text)
    text = re.sub(r"\\s+", " ", text)
    return text.strip()

def tokenizeArabic(text):
    return normalizeText(text).split()

def getSkipBigrams(tokens, maxSkip=4):
    skipBigrams = set()

    for i in range(len(tokens)):
        maxJ = len(tokens) if maxSkip is None else min(len(tokens), i + maxSkip + 2)

        for j in range(i + 1, maxJ):
            skipBigrams.add((tokens[i], tokens[j]))

    return skipBigrams

def rougeSScore(prediction, reference, maxSkip=4):
    predTokens = tokenizeArabic(prediction)
    refTokens = tokenizeArabic(reference)

    predBigrams = getSkipBigrams(predTokens, maxSkip=maxSkip)
    refBigrams = getSkipBigrams(refTokens, maxSkip=maxSkip)

    if len(predBigrams) == 0 or len(refBigrams) == 0:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}

    overlap = len(predBigrams.intersection(refBigrams))
    precision = overlap / len(predBigrams)
    recall = overlap / len(refBigrams)
    f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)

    return {"precision": precision, "recall": recall, "f1": f1}

def computeBertScoreSafe(predictions, references):
    safeIndices = [i for i, prediction in enumerate(predictions) if str(prediction).strip() != ""]
    bertPrecision = torch.zeros(len(predictions))
    bertRecall = torch.zeros(len(predictions))
    bertF1 = torch.zeros(len(predictions))

    if len(safeIndices) > 0:
        safePredictions = [predictions[i] for i in safeIndices]
        safeReferences = [references[i] for i in safeIndices]

        p, r, f1 = bertScore(
            safePredictions,
            safeReferences,
            lang="ar",
            model_type="bert-base-multilingual-cased",
            verbose=True,
            device=deviceForMetrics
        )

        for localIndex, originalIndex in enumerate(safeIndices):
            bertPrecision[originalIndex] = p[localIndex].cpu()
            bertRecall[originalIndex] = r[localIndex].cpu()
            bertF1[originalIndex] = f1[localIndex].cpu()

    return bertPrecision, bertRecall, bertF1

def computeSemanticSimilaritiesSafe(predictions, references):
    similarities = torch.zeros(len(predictions))
    safeIndices = [i for i, prediction in enumerate(predictions) if str(prediction).strip() != ""]

    if len(safeIndices) == 0:
        return similarities

    sentenceModel = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
        device=deviceForMetrics
    )

    safePredictions = [predictions[i] for i in safeIndices]
    safeReferences = [references[i] for i in safeIndices]

    predictionEmbeddings = sentenceModel.encode(
        safePredictions,
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    referenceEmbeddings = sentenceModel.encode(
        safeReferences,
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    safeSimilarities = util.cos_sim(predictionEmbeddings, referenceEmbeddings).diagonal().cpu()

    for localIndex, originalIndex in enumerate(safeIndices):
        similarities[originalIndex] = safeSimilarities[localIndex]

    return similarities

def computeMetrics(predictions, references):
    predictions = [cleanGeneratedText(normalizeText(x)) for x in predictions]
    references = [normalizeText(x) for x in references]

    bleuResult = sacrebleu.corpus_bleu(predictions, [references], force=True)

    bertPrecision, bertRecall, bertF1 = computeBertScoreSafe(predictions, references)
    semanticSimilarities = computeSemanticSimilaritiesSafe(predictions, references)

    rougeScorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)

    rougeLResults = [
        rougeScorer.score(reference, prediction)["rougeL"]
        for prediction, reference in zip(predictions, references)
    ]

    rougeSResults = [
        rougeSScore(prediction, reference, maxSkip=4)
        for prediction, reference in zip(predictions, references)
    ]

    metrics = {
        "BLEU": bleuResult.score,
        "BERTScore_Precision": bertPrecision.mean().item(),
        "BERTScore_Recall": bertRecall.mean().item(),
        "BERTScore_F1": bertF1.mean().item(),
        "Semantic_Textual_Similarity": semanticSimilarities.mean().item(),
        "ROUGE_L_Precision": np.mean([x.precision for x in rougeLResults]),
        "ROUGE_L_Recall": np.mean([x.recall for x in rougeLResults]),
        "ROUGE_L_F1": np.mean([x.fmeasure for x in rougeLResults]),
        "ROUGE_S_Precision": np.mean([x["precision"] for x in rougeSResults]),
        "ROUGE_S_Recall": np.mean([x["recall"] for x in rougeSResults]),
        "ROUGE_S_F1": np.mean([x["f1"] for x in rougeSResults])
    }

    return metrics, semanticSimilarities, rougeLResults, rougeSResults

## 10. Compute and save metrics

In [11]:
resultsDf = pd.read_csv(predictionsPath).fillna("").sort_values("rowIndex")

predictions = resultsDf["prediction"].astype(str).tolist()
referencesForMetrics = resultsDf["referenceAnswer"].astype(str).tolist()

errorCount = sum(resultsDf["error"].astype(str).str.strip() != "")
emptyPredictionCount = sum(resultsDf["prediction"].astype(str).str.strip() == "")

print("Rows:", len(resultsDf))
print("Errors:", errorCount)
print("Empty predictions:", emptyPredictionCount)

openaiMetrics, openaiSemanticSimilarities, openaiRougeLResults, openaiRougeSResults = computeMetrics(
    predictions,
    referencesForMetrics
)

openaiMetricsDf = pd.DataFrame([{
    "system": f"openai_{openaiModelName}",
    "num_examples": len(resultsDf),
    "num_errors": errorCount,
    "num_empty_predictions": emptyPredictionCount,
    **openaiMetrics
}])

openaiMetricsDf.to_csv(metricsPath, index=False, encoding="utf-8-sig")

resultsDf["cleanedPrediction"] = [cleanGeneratedText(prediction) for prediction in predictions]
resultsDf["semanticSimilarity"] = openaiSemanticSimilarities.cpu().numpy()
resultsDf["rougeL_F1"] = [x.fmeasure for x in openaiRougeLResults]
resultsDf["rougeS_F1"] = [x["f1"] for x in openaiRougeSResults]

resultsDf.to_csv(predictionsPath, index=False, encoding="utf-8-sig")

print(openaiMetricsDf.T)
print("\\nSaved metrics to:", metricsPath)
print("Updated predictions saved to:", predictionsPath)

Rows: 1571
Errors: 0
Empty predictions: 0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (18 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

                                          0
system                       openai_gpt-5.4
num_examples                           1571
num_errors                                0
num_empty_predictions                     0
BLEU                                0.90721
BERTScore_Precision                0.690427
BERTScore_Recall                   0.598685
BERTScore_F1                       0.640271
Semantic_Textual_Similarity         0.45205
ROUGE_L_Precision                  0.047479
ROUGE_L_Recall                     0.007837
ROUGE_L_F1                         0.012737
ROUGE_S_Precision                  0.030954
ROUGE_S_Recall                     0.017247
ROUGE_S_F1                         0.019999
\nSaved metrics to: openai_chatgpt_evaluation/gpt-5.4_heldout_metrics.csv
Updated predictions saved to: openai_chatgpt_evaluation/gpt-5.4_heldout_predictions.csv


## 11. Inspect worst and best examples

In [12]:
displayColumns = [
    "rowIndex",
    "referenceAnswer",
    "prediction",
    "cleanedPrediction",
    "semanticSimilarity",
    "rougeL_F1",
    "rougeS_F1",
    "error"
]

print("Lowest ROUGE-L examples:")
display(resultsDf.sort_values("rougeL_F1")[displayColumns].head(10))

print("Highest ROUGE-L examples:")
display(resultsDf.sort_values("rougeL_F1", ascending=False)[displayColumns].head(10))

Lowest ROUGE-L examples:


,rowIndex,referenceAnswer,prediction,cleanedPrediction,semanticSimilarity,rougeL_F1,rougeS_F1,error
1025,1025,"واضاف مخامرة: ""لقد صادر جنود الاحتلال الجرارات...",النتيجة التي وردت هي تصاعد معاناة الأهالي، عبر...,النتيجة التي وردت هي تصاعد معاناة الأهالي، عبر...,0.410650,0.0,0.000000,
1039,1039,قُتل أربعة عشر يهودياً، واستشهد اثنان من المسم...,نتيجة معركة منطقة السدرة قرب المسمية الكبيرة ك...,نتيجة معركة منطقة السدرة قرب المسمية الكبيرة ك...,0.492923,0.0,0.000000,
1037,1037,من فئة الريسا ذوي النفوذ الكبير في أعمال الموا...,لا تتوفر معلومات كافية.,لا تتوفر معلومات كافية.,0.159513,0.0,0.000000,
1036,1036,كان الناس يسهرون على السطوح حتى الصباح يضحكون ...,كانت السهرات الاجتماعية في كويكات بسيطة ودافئة...,كانت السهرات الاجتماعية في كويكات بسيطة ودافئة...,0.590396,0.0,0.005495,
1035,1035,بدأ طلب العلم في غزة على يد علماء محليين، ثم ا...,ارتبط طلب العلم في غزة بـالكتاتيب والمدارس وال...,ارتبط طلب العلم في غزة بـالكتاتيب والمدارس وال...,0.759589,0.0,0.053030,
1034,1034,نعم، ما زال حوش الجيطان قائماً ومسكوناً في الب...,نعم، بحسب الروايات المحلية المتداولة عن البلدة...,نعم، بحسب الروايات المحلية المتداولة عن البلدة...,0.725586,0.0,0.038462,
1033,1033,وجد في صفورية لحامون، بائع قماش، خياطات، دايات...,قبل النكبة، وُجدت في صفورية مهن متعددة، أبرزها...,قبل النكبة، وُجدت في صفورية مهن متعددة، أبرزها...,0.534773,0.0,0.008163,
1032,1032,"على رأس طلعة ""عبد شاكر""، وبالقرب من موقف تكسيا...",القضية الأساسية في شارع الشهداء في الخليل هي إ...,القضية الأساسية في شارع الشهداء في الخليل هي إ...,0.710692,0.0,0.012371,
1031,1031,قامت مجموعة من المستوطنين الاسرائيليين، وتحت ح...,لا تتوفر معلومات كافية.,لا تتوفر معلومات كافية.,0.089782,0.0,0.000000,
1030,1030,كانت تستخدم للأغنام ومطاعم ومسبح للأطفال وغيره...,كانت المنشآت المهدومة في العيسوية تُستخدم للسك...,كانت المنشآت المهدومة في العيسوية تُستخدم للسك...,0.460393,0.0,0.000000,


Highest ROUGE-L examples:


,rowIndex,referenceAnswer,prediction,cleanedPrediction,semanticSimilarity,rougeL_F1,rougeS_F1,error
1496,1496,من سنين 1948 إلى حد سنة 1986 كان في حكم عسكري ...,الحكم العسكري الذي كان سائداً هو نظام الحكم ال...,الحكم العسكري الذي كان سائداً هو نظام الحكم ال...,0.757924,0.666667,0.000000,
853,853,ساهم الأهالي بدفع 22 ألف ليرة لمد الخط الرئيسي...,دفع أهالي معاوية لمشروع المياه عام 1973-1974 م...,دفع أهالي معاوية لمشروع المياه عام 1973-1974 م...,0.694998,0.666667,0.000000,
1409,1409,اقتحم مستوطنون متطرفون صباح الثلاثاء 17/6/2014...,في 17/6/2014 اقتحم مستوطنون المسجد الأقصى المب...,في 17/6/2014 اقتحم مستوطنون المسجد الأقصى المب...,0.825731,0.461538,0.054054,
914,914,قدموا الشهيد عبد العزيز أبو زرار في معركة احتل...,الحدث المرتبط بعائلة أبو زرار في معركة احتلال ...,الحدث المرتبط بعائلة أبو زرار في معركة احتلال ...,0.730696,0.461538,0.193548,
629,629,اقتحم مستوطنون متطرفون صباح الثلاثاء 17/6/2014...,في 17/6/2014 جرت اقتحامات للمسجد الأقصى من جهة...,في 17/6/2014 جرت اقتحامات للمسجد الأقصى من جهة...,0.888116,0.461538,0.060606,
1038,1038,صدر عن **الحاكم العسكري العام** لقرى المثلث، و...,في 21 أكتوبر 1949 صدر عن الحاكم العسكري العام ...,في 21 أكتوبر 1949 صدر عن الحاكم العسكري العام ...,0.708216,0.444444,0.018882,
1286,1286,قامت جرافات البلدية صباح اليوم السبت 7/6/2014،...,بتاريخ 7/6/2014، هدمت سلطات الاحتلال في الرملة...,بتاريخ 7/6/2014، هدمت سلطات الاحتلال في الرملة...,0.834137,0.428571,0.000000,
697,697,اقتلع عدد من المستوطنين مساء الأحد 8/9/2013 أك...,التفاصيل المرتبطة بالضفة الغربية في هذه القضية...,التفاصيل المرتبطة بالضفة الغربية في هذه القضية...,0.858426,0.428571,0.150376,
1549,1549,هدمت جرافات الاحتلال منزل عائلة صالحية فجر الأ...,هُدم منزل عائلة صالحية في حي الشيخ جراح في 19 ...,هُدم منزل عائلة صالحية في حي الشيخ جراح في 19 ...,0.736659,0.363636,0.140625,
1410,1410,بعد وقفة أهل الطيرة في معارك عام 1948، سُلّمت ...,بعد معارك عام 1948 بقيت الطيرة صامدة فترة، ثم ...,بعد معارك عام 1948 بقيت الطيرة صامدة فترة، ثم ...,0.645256,0.363636,0.019672,
